In [4]:
pip install senticnet pandas scikit-learn openpyxl emoji contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 19.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [contractions] [anyascii]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Normalisation functions

In [5]:
import re
import string
import emoji
import contractions

def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def normalise_hashtags(text):
    # #happy -> happy
    return re.sub(r'#(\w+)', r'\1', text)

def normalise_emoji(text):
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    return text

def expand_contractions(text):
    # can't -> cannot, it's -> it is
    return contractions.fix(text)

def normalise_elongated(text):
    # soo -> so, goooood -> good-ish
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def clean_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def normalise_microtext(text):
    text = str(text)

    text = text.lower()

    text = remove_urls(text)
    text = remove_mentions(text)
    text = normalise_hashtags(text)
    text = normalise_emoji(text)
    text = expand_contractions(text)
    text = normalise_elongated(text)
    text = remove_punctuation(text)
    text = clean_whitespace(text)

    return text

### Classification & Evaluation

In [6]:
import pandas as pd
import time
from senticnet.senticnet import SenticNet
from sklearn.metrics import classification_report

# Initialize SenticNet
sn = SenticNet()

def classify_with_senticnet(text):
    """
    Returns (subjectivity_label, polarity_label)
    Subjectivity: 0 = Objective, 1 = Subjective
    Polarity: 0 = Negative, 1 = Positive
    """
    normalised_text = normalise_microtext(text)
    words = normalised_text.split()
    total_polarity = 0
    match_count = 0
    
    for word in words:
        try:
            # SenticNet lookup
            val = float(sn.polarity_value(word))
            total_polarity += val
            match_count += 1
        except KeyError:
            continue # Word not in knowledge base
            
    # Subtask 1: Subjectivity Detection
    subjectivity = 1 if match_count > 0 else 0
    
    # Subtask 2: Polarity Detection
    # If neutral or sum is 0, we default to Negative or Neutral based on your project needs
    polarity = 1 if total_polarity > 0 else 0
    
    return subjectivity, polarity

def run_q4_evaluation():
    # Load your manually labeled data
    df = pd.read_excel('eval.xlsx') # Required: 'text', 'gt_subjectivity', 'gt_polarity'
    
    results_subj = []
    results_pol = []
    
    # Measure Performance (Requirement Q4)
    start_time = time.time()
    
    for index, row in df.iterrows():
        subj, pol = classify_with_senticnet(row['text'])
        results_subj.append(subj)
        results_pol.append(pol)
        
    end_time = time.time()
    
    # --- EVALUATION METRICS ---
    
    print("--- Subtask 1: Subjectivity Detection (SenticNet) ---")
    print(classification_report(df['gt_subjectivity'], results_subj, target_names=['Objective', 'Subjective'], zero_division=0))
    
    print("\n--- Subtask 2: Polarity Detection (SenticNet) ---")
    # Only evaluate polarity on records that were actually subjective
    subj_indices = [i for i, x in enumerate(df['gt_subjectivity']) if x == 1]
    y_true_pol = [df['gt_polarity'].iloc[i] for i in subj_indices]
    y_pred_pol = [results_pol[i] for i in subj_indices]
    
    print(classification_report(y_true_pol, y_pred_pol, target_names=['Negative', 'Positive'], zero_division=0))
    
    # Calculate Speed (Requirement Q4)
    duration = end_time - start_time
    records_per_sec = len(df) / duration
    print(f"\nProcessing Speed: {records_per_sec:.2f} records/second")



In [7]:
run_q4_evaluation()

--- Subtask 1: Subjectivity Detection (SenticNet) ---
              precision    recall  f1-score   support

   Objective       0.00      0.00      0.00         0
  Subjective       1.00      0.87      0.93      1000

    accuracy                           0.87      1000
   macro avg       0.50      0.43      0.46      1000
weighted avg       1.00      0.87      0.93      1000


--- Subtask 2: Polarity Detection (SenticNet) ---
              precision    recall  f1-score   support

    Negative       0.69      0.50      0.58       500
    Positive       0.61      0.77      0.68       500

    accuracy                           0.64      1000
   macro avg       0.65      0.64      0.63      1000
weighted avg       0.65      0.64      0.63      1000


Processing Speed: 6396.70 records/second
